# Experiment 14 — AutoGluon + AutoGluon residual stack

Layer 1 is the Exp 9 GBM + CatBoost AutoGluon ensemble. Layer 2 is another AutoGluon GBM + CatBoost ensemble trained to predict Layer 1's chronological OOF residuals from the same original best features. Both layers use AutoGluon's usual categorical handling. Final forecast = Layer 1 forecast + Layer 2 residual forecast.

In [1]:
from pathlib import Path
import json,os,sys
import numpy as np
import pandas as pd
from autogluon.tabular import TabularPredictor
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
from sklearn.model_selection import TimeSeriesSplit
configured_root=os.environ.get('KDDM_PROJECT_ROOT')
roots=([Path(configured_root)] if configured_root else [])+[Path.cwd(),*Path.cwd().parents]
project_root=next((root for root in roots if (root/'exploration/metul/preprocessing.py').exists()),None)
if project_root is None: raise FileNotFoundError('Set KDDM_PROJECT_ROOT to the project directory.')
sys.path.insert(0,str(project_root/'exploration'/'metul'))
from preprocessing import RentalPreprocessor,load_datasets
TARGET='MONTHLY_RENT'; RESIDUAL_TARGET='layer1_residual'; N_SPLITS=3
OOF_TIME_LIMIT=int(os.environ.get('RESIDUAL_OOF_TIME_LIMIT','180'))
LAYER1_TIME_LIMIT=int(os.environ.get('AUTOGLUON_TIME_LIMIT','600'))
LAYER2_TIME_LIMIT=int(os.environ.get('RESIDUAL_LAYER2_TIME_LIMIT','300'))
RUN_ID=os.environ.get('AUTOGLUON_RUN_ID',pd.Timestamp.now().strftime('%Y%m%d_%H%M%S'))
HYPERPARAMETERS={'GBM':{},'CAT':{}}
source_nb=json.loads((project_root/'exploration'/'metul'/'exp_14_resale_features_lgbm.ipynb').read_text())
source_cells={cell.get('metadata',{}).get('id'):''.join(cell.get('source',[])) for cell in source_nb['cells']}
exec(source_cells['setup'],globals())
exec(source_cells['exp9-best-feature-transformer'],globals())


In [2]:
_,datasets=load_datasets(project_root/'data')
raw=datasets['train'].copy();raw['period']=pd.to_datetime(raw['RENT_APPROVAL_DATE']+'-01').dt.to_period('M')
periods=np.sort(raw.period.unique());cutoff=periods[int(len(periods)*.80)]
raw_fit,raw_valid=raw.loc[raw.period<cutoff].copy(),raw.loc[raw.period>=cutoff].copy()
def best_features(history,future):
    processor=RentalPreprocessor(datasets)
    history_base=processor.fit_transform(history.drop(columns=['period',TARGET],errors='ignore'))
    future_base=processor.transform(future.drop(columns=['period',TARGET],errors='ignore'))
    return make_exp9_best_features(history,future,history_base,future_base)
fit_x,valid_x=best_features(raw_fit,raw_valid)
fit_y=raw_fit[TARGET].astype(float).reset_index(drop=True);valid_y=raw_valid[TARGET].astype(float).reset_index(drop=True)
print(f'Cutoff={cutoff}; fit={fit_x.shape}; validation={valid_x.shape}')


[categorical_clean] HDB lookup | TOWN: 26 -> 26 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | BLOCK: 2735 -> 2735 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | STREET: 576 -> 576 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | POSTAL_CODE: 9180 -> 9180 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | SUBZONE: 156 -> 156 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | PLANNING_AREA: 31 -> 31 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces


In [3]:
def fit_autogluon(train_x,train_y,label,path,time_limit):
    return TabularPredictor(label=label,problem_type='regression',eval_metric='root_mean_squared_error',path=str(path)).fit(train_data=train_x.assign(**{label:train_y.to_numpy()}),hyperparameters=HYPERPARAMETERS,presets=['medium_quality','ignore_text'],time_limit=time_limit,verbosity=2)
fit_periods=np.sort(raw_fit.period.unique());oof_parts=[]
for fold,(early_index,later_index) in enumerate(TimeSeriesSplit(n_splits=N_SPLITS).split(fit_periods),1):
    early=raw_fit.loc[raw_fit.period.isin(fit_periods[early_index])].copy();later=raw_fit.loc[raw_fit.period.isin(fit_periods[later_index])].copy()
    early_x,later_x=best_features(early,later)
    model=fit_autogluon(early_x,early[TARGET].astype(float).reset_index(drop=True),TARGET,project_root/'exploration'/'metul'/'ag_exp_14_ag_residual_models'/RUN_ID/f'layer1_oof_{fold}',OOF_TIME_LIMIT)
    part=later_x.copy();part[RESIDUAL_TARGET]=later[TARGET].to_numpy()-model.predict(later_x).astype(float).to_numpy();part['fold']=fold
    oof_parts.append(part);print(f'Layer 1 OOF fold {fold}: fit={len(early):,}; OOF={len(later):,}')
oof=pd.concat(oof_parts,ignore_index=True)


[categorical_clean] HDB lookup | TOWN: 26 -> 26 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | BLOCK: 2735 -> 2735 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | STREET: 576 -> 576 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | POSTAL_CODE: 9180 -> 9180 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | SUBZONE: 156 -> 156 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | PLANNING_AREA: 31 -> 31 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces


Verbosity: 2 (Standard Logging)


[categorical_clean] derived HDB feature | REGION: 5 -> 5 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces


=================== System Info ===================
AutoGluon Version:  1.6.2
Python Version:     3.12.14
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.6.0: Fri Jul 31 19:16:20 PDT 2026; root:xnu-12377.161.14~5/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       5.09 GB / 16.00 GB (31.8%)
Disk Space Avail:   168.65 GB / 460.38 GB (36.6%)
Presets specified: ['medium_quality', 'ignore_text']
Beginning AutoGluon training ... Time limit = 180s
AutoGluon will save models to "/Users/metul/Desktop/KDDM-project/exploration/metul/ag_exp_14_ag_residual_models/20261015_144824/layer1_oof_1"
Train Data Rows:    31565
Train Data Columns: 18
Label Column:       MONTHLY_RENT
Problem Type:       regression
Preprocessing data...
Using Feature Generators to preprocess the data ...
F

Layer 1 OOF fold 1: fit=31,565; OOF=28,514
[categorical_clean] HDB lookup | TOWN: 26 -> 26 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | BLOCK: 2735 -> 2735 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | STREET: 576 -> 576 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | POSTAL_CODE: 9180 -> 9180 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | SUBZONE: 156 -> 156 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | PLANNING_AREA: 31 -> 31 unique values | lowercase, trim, collapse whitespace, and replace hyph

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.2
Python Version:     3.12.14
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.6.0: Fri Jul 31 19:16:20 PDT 2026; root:xnu-12377.161.14~5/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       5.08 GB / 16.00 GB (31.7%)
Disk Space Avail:   168.64 GB / 460.38 GB (36.6%)
Presets specified: ['medium_quality', 'ignore_text']
Beginning AutoGluon training ... Time limit = 180s
AutoGluon will save models to "/Users/metul/Desktop/KDDM-project/exploration/metul/ag_exp_14_ag_residual_models/20261015_144824/layer1_oof_2"
Train Data Rows:    60079
Train Data Columns: 18
Label Column:       MONTHLY_RENT
Problem Type:       regression
Preprocessing data...
Using Feature Generat

Layer 1 OOF fold 2: fit=60,079; OOF=27,858
[categorical_clean] HDB lookup | TOWN: 26 -> 26 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | BLOCK: 2735 -> 2735 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | STREET: 576 -> 576 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | POSTAL_CODE: 9180 -> 9180 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | SUBZONE: 156 -> 156 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | PLANNING_AREA: 31 -> 31 unique values | lowercase, trim, collapse whitespace, and replace hyph

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.2
Python Version:     3.12.14
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.6.0: Fri Jul 31 19:16:20 PDT 2026; root:xnu-12377.161.14~5/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       5.08 GB / 16.00 GB (31.7%)
Disk Space Avail:   168.62 GB / 460.38 GB (36.6%)
Presets specified: ['medium_quality', 'ignore_text']
Beginning AutoGluon training ... Time limit = 180s
AutoGluon will save models to "/Users/metul/Desktop/KDDM-project/exploration/metul/ag_exp_14_ag_residual_models/20261015_144824/layer1_oof_3"
Train Data Rows:    87937
Train Data Columns: 18
Label Column:       MONTHLY_RENT
Problem Type:       regression
Preprocessing data...
Using Feature Generat

Layer 1 OOF fold 3: fit=87,937; OOF=30,051


In [4]:
# Preserve original categorical columns; AutoGluon's feature generator handles them in Layer 2.
residual_x=oof.drop(columns=[RESIDUAL_TARGET,'fold'])
layer1=fit_autogluon(fit_x,fit_y,TARGET,project_root/'exploration'/'metul'/'ag_exp_14_ag_residual_models'/RUN_ID/'layer1_full',LAYER1_TIME_LIMIT)
layer1_valid=layer1.predict(valid_x).astype(float).clip(lower=0)


Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.2
Python Version:     3.12.14
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.6.0: Fri Jul 31 19:16:20 PDT 2026; root:xnu-12377.161.14~5/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       5.24 GB / 16.00 GB (32.8%)
Disk Space Avail:   168.58 GB / 460.38 GB (36.6%)
Presets specified: ['medium_quality', 'ignore_text']
Beginning AutoGluon training ... Time limit = 600s
AutoGluon will save models to "/Users/metul/Desktop/KDDM-project/exploration/metul/ag_exp_14_ag_residual_models/20261015_144824/layer1_full"
Train Data Rows:    117988
Train Data Columns: 18
Label Column:       MONTHLY_RENT
Problem Type:       regression
Preprocessing data...
Using Feature Generat

In [5]:
layer2=fit_autogluon(residual_x,oof[RESIDUAL_TARGET],RESIDUAL_TARGET,project_root/'exploration'/'metul'/'ag_exp_14_ag_residual_models'/RUN_ID/'layer2_residual',LAYER2_TIME_LIMIT)
residual_valid=layer2.predict(valid_x).astype(float)
final_prediction=np.clip(layer1_valid.to_numpy()+residual_valid.to_numpy(),a_min=0,a_max=None)
def metric(name,prediction):return {'model':name,'RMSE':mean_squared_error(valid_y,prediction)**.5,'MAE':mean_absolute_error(valid_y,prediction),'R2':r2_score(valid_y,prediction)}
comparison=pd.DataFrame([metric('Layer 1 AutoGluon',layer1_valid),metric('Layer 1 + Layer 2 AutoGluon residual',final_prediction)])
display(comparison.sort_values('RMSE'))
display(pd.DataFrame({'OOF_residual':oof[RESIDUAL_TARGET],'layer2_fitted_residual':layer2.predict(residual_x)}).describe())


Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.2
Python Version:     3.12.14
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.6.0: Fri Jul 31 19:16:20 PDT 2026; root:xnu-12377.161.14~5/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       5.14 GB / 16.00 GB (32.1%)
Disk Space Avail:   168.51 GB / 460.38 GB (36.6%)
Presets specified: ['medium_quality', 'ignore_text']
Beginning AutoGluon training ... Time limit = 300s
AutoGluon will save models to "/Users/metul/Desktop/KDDM-project/exploration/metul/ag_exp_14_ag_residual_models/20261015_144824/layer2_residual"
Train Data Rows:    86423
Train Data Columns: 18
Label Column:       layer1_residual
Problem Type:       regression
Preprocessing data...
Using Feature G

,model,RMSE,MAE,R2
0,Layer 1 AutoGluon,468.271759,340.986810,0.491986
1,Layer 1 + Layer 2 AutoGluon residual,473.949351,338.281684,0.479593


,OOF_residual,layer2_fitted_residual
count,86423.000000,86423.000000
mean,217.050004,218.067093
std,572.729411,189.308273
min,-3873.494629,-452.371429
25%,-137.619995,71.065884
50%,201.413574,159.668503
75%,584.700317,351.363953
max,3838.455078,2294.392334


## Tune the residual correction weight

The final forecast is `Layer 1 + weight × Layer 2 residual`. This selects the weight minimizing validation RMSE, including `0` (no residual correction) and `1` (the unweighted stack).

In [7]:
# Closed-form least-squares optimum, then a nearby grid for an auditable RMSE choice.
base_prediction = layer1_valid.to_numpy()
residual_prediction = residual_valid.to_numpy()
actual = valid_y.to_numpy()
denominator = np.dot(residual_prediction, residual_prediction)
analytic_weight = 0.0 if denominator == 0 else np.dot(actual - base_prediction, residual_prediction) / denominator
weight_grid = np.unique(np.concatenate([np.linspace(-1.0, 2.0, 301), [analytic_weight, 0.0, 1.0]]))
weight_results = pd.DataFrame({
    'residual_weight': weight_grid,
    'RMSE': [mean_squared_error(actual, np.clip(base_prediction + weight * residual_prediction, a_min=0, a_max=None)) ** 0.5 for weight in weight_grid],
})
best_weight = float(weight_results.loc[weight_results['RMSE'].idxmin(), 'residual_weight'])
weighted_final_prediction = np.clip(base_prediction + best_weight * residual_prediction, a_min=0, a_max=None)
display(weight_results.nsmallest(10, 'RMSE'))
display(pd.DataFrame([
    metric('Layer 1 only (weight=0)', base_prediction),
    metric('Unweighted residual stack (weight=1)', np.clip(base_prediction + residual_prediction, a_min=0, a_max=None)),
    metric(f'Weighted residual stack (weight={best_weight:.3f})', weighted_final_prediction),
]))
print(f'Analytic residual weight: {analytic_weight:.6f}; selected RMSE weight: {best_weight:.6f}')


,residual_weight,RMSE
116,0.159928,468.056909
117,0.160000,468.056909
115,0.150000,468.057737
118,0.170000,468.057762
114,0.140000,468.060246
119,0.180000,468.060294
113,0.130000,468.064435
120,0.190000,468.064508
112,0.120000,468.070304
121,0.200000,468.070401


,model,RMSE,MAE,R2
0,Layer 1 only (weight=0),468.271759,340.986810,0.491986
1,Unweighted residual stack (weight=1),473.949351,338.281684,0.479593
2,Weighted residual stack (weight=0.160),468.056909,339.429918,0.492452


Analytic residual weight: 0.159928; selected RMSE weight: 0.159928


In [10]:
# Export the best observed configuration: base AutoGluon + two weighted residual layers.
BEST_RESIDUAL_LAYER_COUNT = 2
assert len(stage_test_predictions) > BEST_RESIDUAL_LAYER_COUNT

best_two_residual_submission = pd.DataFrame({
    'Id': production_test.index,
    'Predicted': stage_test_predictions[BEST_RESIDUAL_LAYER_COUNT],
})
assert (
    best_two_residual_submission['Id'].is_unique
    and best_two_residual_submission['Predicted'].notna().all()
    and len(best_two_residual_submission) == len(production_test)
)

best_two_residual_submission_path = (
    project_root / 'exploration' / 'metul'
    / 'exp_14_autogluon_two_residual_layers_submission.csv'
)
best_two_residual_submission.to_csv(
    best_two_residual_submission_path, index=False
)
print(
    f'Saved {len(best_two_residual_submission):,} predictions using residual weights '
    f'{[round(weight, 6) for weight in residual_weights[:BEST_RESIDUAL_LAYER_COUNT]]} '
    f'to {best_two_residual_submission_path}'
)
display(best_two_residual_submission.head())

Saved 50,000 predictions using residual weights [0.505861, 0.201996] to /Users/metul/Desktop/KDDM-project/exploration/metul/exp_14_autogluon_two_residual_layers_submission.csv


,Id,Predicted
0,0,2733.529131
1,1,3708.380021
2,2,3296.163685
3,3,3140.513679
4,4,3282.990207
